In [4]:
from document_extractor import extract_itac_report

doc_1_path = "/Users/afschowdhury/Code Local/itac-report-validator/docs/report1/LS2502 - Final Draft R2.docx"
doc_2_path = "/Users/afschowdhury/Code Local/itac-report-validator/docs/report2/LS2508 - Final Draft.docx"
out = extract_itac_report(doc_2_path, output="html", save_files=True)

rec_summary_table_html = out["recommendation_summary_table"]  # HTML for Table 1-3
print(rec_summary_table_html)  # or write it to a file

<table border='1' cellpadding='4' cellspacing='0' style='border-collapse:collapse;width:100%'><tr><td><p style="text-align:center"><b>AR No.</b></p></td><td><p style="text-align:center"><b>Category</b></p></td><td><p style="text-align:center"><b>Description</b></p></td><td><p style="text-align:center"><b>Electricity Savings (kWh/yr)</b></p></td><td><p style="text-align:center"><b>Energy Cost Savings ($/yr)</b></p></td><td><p style="text-align:center"><b>Demand Savings</b></p><p style="text-align:center"><b>(kW/yr)</b></p></td><td><p style="text-align:center"><b>Demand</b></p><p style="text-align:center"><b>Cost Savings ($/yr)</b></p></td><td><p style="text-align:center"><b>Admin Cost Savings</b></p><p style="text-align:center"><b>($/yr)</b></p></td><td><p style="text-align:center"><b>Propane Savings</b></p><p style="text-align:center"><b>(mmbtu/yr)</b></p></td><td><p style="text-align:center"><b>Propane</b></p><p style="text-align:center"><b>Cost Saving</b></p><p style="text-align:cent

In [5]:
from IPython.display import HTML

In [6]:
out['ar_summary']

'<p><i>AR No. 1 –\xa0</i><i>Utilize Higher Efficiency Lamps and/or Ballasts</i></p>\n<p>Utilizing higher-efficiency lamps and/or ballasts will enhance lighting system performance while reducing energy consumption. This energy efficiency measure will yield $771 in total cost savings and incur an implementation cost of $675. The project has a payback period of 0.88 years and will reduce CO₂ emissions by 3 tons annually, with annual energy savings of 6,806 kWh.</p>\n<p><i>AR No. 2 –\xa0</i><i>Install Sub-metering Equipment</i></p>\n<p>Installing sub-metering equipment will enable better monitoring and management of electrical energy consumption. This energy efficiency measure will yield $3,313 in total cost savings and incur an implementation cost of $3,000. The project has a payback period of 0.91 years and will reduce CO₂ emissions by 12 tons annually, with annual energy savings of 32,484 kWh.</p>\n<p><i>AR No. 3 – </i><i>Modify Inventory Control</i></p>\n<p>Modifying inventory control 

In [7]:
display(HTML(out['ar_summary']))

In [8]:
display(HTML(rec_summary_table_html))

AR No.,Category,Description,Electricity Savings (kWh/yr),Energy Cost Savings ($/yr),Demand Savings(kW/yr),DemandCost Savings ($/yr),Admin Cost Savings($/yr),Propane Savings(mmbtu/yr),PropaneCost Saving($/yr),Total Cost Savings ($/yr),CO2 Reduction (Tons/yr),Impl.Cost ($),Payback Period(yrs)
1,Lighting,Utilize Higher Efficiency Lamps and/or Ballasts,"6,806",694,17,77,0,0,0,771,3,675,0.88
2,Administrative,Install Sub-metering Equipment,"32,484","3,313",0,0,0,0,0,"3,313",12,"3,000",0.91
3,Inventory,Modify Inventory Control,0,0,0,0,"9,000",0,0,"9,000",0,"25,000",2.78
4,Space Conditioning,Replace Existing HVAC with Higher Efficiency Model,"14,586","1,488",0,0,0,0,0,"1,488",6,"5,190",3.49
5,Alternative Energy use,Use Solar Heat to Generate Electricity,"387,960","39,572",0,0,0,0,0,"39,572",148,"160,992",4.07
6,Motors,Consider Replacement of Old Motors with Energy-Efficient Ones,"24,612","2,510",0,0,0,0,0,"2,510",9,"10,600",4.22
7,Air Compressors,Purchase Optimum Sized Air Compressor with More Suitable Substitutes,"36,365","3,709",0,0,0,0,0,"3,709",14,"16,949",4.57
8,Fuel Switching,Replace Fossil Fuel Equipment with Electrical Equipment,"-3,744",-382,0,0,"8,000",38,"1,045","8,663",1,"48,183",5.56
,,TOTALS,"100,267","10,226",17,77,"17,000",38,"1,045","69,026",193,"270,589",3.92


In [16]:
def parse_recommendation_table_to_json(table_html):
    """
    Parse the recommendation summary table HTML and return a list of dictionaries 
    containing each recommendation's details.
    
    Args:
        table_html (str): HTML string of the recommendation summary table
        
    Returns:
        List[Dict]: List of dictionaries, each containing recommendation details
    """
    from bs4 import BeautifulSoup
    import re
    
    if not table_html:
        return []
    
    soup = BeautifulSoup(table_html, 'html.parser')
    table = soup.find('table')
    
    if not table:
        return []
    
    rows = table.find_all('tr')
    if len(rows) < 2:  # Need at least header + 1 data row
        return []
    
    # Extract header row to identify columns
    header_row = rows[0]
    headers = [th.get_text(strip=True) for th in header_row.find_all(['th', 'td'])]
    
    recommendations = []
    
    # Process data rows
    for row in rows[1:]:
        cells = row.find_all(['td', 'th'])
        if len(cells) < len(headers):
            continue
            
        recommendation = {}
        
        for i, cell in enumerate(cells):
            if i >= len(headers):
                break
                
            header = headers[i].lower()
            cell_text = cell.get_text(strip=True)
            
            # Map common header variations to standardized keys
            if 'ar' in header and 'no' in header:
                # Try to convert ar_number to float if it contains numbers
                number_match = re.search(r'([\d.]+)', cell_text)
                recommendation['ar_number'] = float(number_match.group(1)) if number_match else cell_text
            elif 'recommendation' in header or 'description' in header:
                recommendation['description'] = cell_text
            elif 'category' in header:
                recommendation['category'] = cell_text
            elif 'cost' in header and 'saving' in header:
                # Extract numeric value from cost savings
                cost_match = re.search(r'[\$]?([\d,]+)', cell_text.replace(',', ''))
                recommendation['cost_savings'] = float(cost_match.group(1).replace(',', '')) if cost_match else 0.0
            elif 'implementation' in header and 'cost' in header:
                # Extract numeric value from implementation cost
                cost_match = re.search(r'[\$]?([\d,]+)', cell_text.replace(',', ''))
                recommendation['implementation_cost'] = float(cost_match.group(1).replace(',', '')) if cost_match else 0.0
            elif 'payback' in header:
                # Extract numeric value from payback period
                payback_match = re.search(r'([\d.]+)', cell_text)
                recommendation['payback_period_years'] = float(payback_match.group(1)) if payback_match else 0.0
            elif 'energy' in header and 'saving' in header:
                # Extract numeric value from energy savings
                energy_match = re.search(r'([\d,]+)', cell_text.replace(',', ''))
                recommendation['energy_savings_kwh'] = float(energy_match.group(1).replace(',', '')) if energy_match else 0.0
            elif 'co2' in header or 'emission' in header:
                # Extract numeric value from CO2 reduction
                co2_match = re.search(r'([\d,]+)', cell_text.replace(',', ''))
                recommendation['co2_reduction_tons'] = float(co2_match.group(1).replace(',', '')) if co2_match else 0.0
            else:
                # Use the header as-is for any other columns
                key = re.sub(r'[^\w\s]', '', header).replace(' ', '_')
                # Try to convert to float if it contains numbers, unless it's category or description
                if key not in ['description', 'category']:
                    number_match = re.search(r'([\d,.-]+)', cell_text)
                    if number_match:
                        try:
                            recommendation[key] = float(number_match.group(1).replace(',', ''))
                        except ValueError:
                            recommendation[key] = cell_text
                    else:
                        recommendation[key] = cell_text
                else:
                    recommendation[key] = cell_text
        
        if recommendation:  # Only add if we got some data
            recommendations.append(recommendation)
    
    return recommendations
                else:
                    recommendation[key] = cell_text
        
        if recommendation:  # Only add if we got some data
            recommendations.append(recommendation)
    
    return recommendations
# Test the function
recommendations_json = parse_recommendation_table_to_json(rec_summary_table_html)
print(f"Found {len(recommendations_json)} recommendations:")
for i, rec in enumerate(recommendations_json, 1):
    print(f"\nRecommendation {i}:")
    for key, value in rec.items():
        print(f"  {key}: {value}")



IndentationError: unindent does not match any outer indentation level (<tokenize>, line 101)

In [12]:
recommendations_json

[{'ar_number': '1',
  'category': 'Lighting',
  'description': 'Utilize Higher Efficiency Lamps and/or Ballasts',
  'electricity_savings_kwhyr': 6806.0,
  'cost_savings': 771.0,
  'demand_savingskwyr': 17.0,
  'propane_savingsmmbtuyr': 0.0,
  'co2_reduction_tons': 3.0,
  'implcost_': 675.0,
  'payback_period_years': 0.88},
 {'ar_number': '2',
  'category': 'Administrative',
  'description': 'Install Sub-metering Equipment',
  'electricity_savings_kwhyr': 32484.0,
  'cost_savings': 3313.0,
  'demand_savingskwyr': 0.0,
  'propane_savingsmmbtuyr': 0.0,
  'co2_reduction_tons': 12.0,
  'implcost_': 3000.0,
  'payback_period_years': 0.91},
 {'ar_number': '3',
  'category': 'Inventory',
  'description': 'Modify Inventory Control',
  'electricity_savings_kwhyr': 0.0,
  'cost_savings': 9000.0,
  'demand_savingskwyr': 0.0,
  'propane_savingsmmbtuyr': 0.0,
  'co2_reduction_tons': 0.0,
  'implcost_': 25000.0,
  'payback_period_years': 2.78},
 {'ar_number': '4',
  'category': 'Space Conditioning',
